In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from rockphypy import QI, EM

plt.rcParams['font.size'] = 14
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.labelpad'] = 10.0

# parameters
Dqz, Kqz, Gqz = 2.65, 36.6, 45  # grain density, bulk and shear modulus
Dsh, Ksh, Gsh = 2.7, 21, 7      # shale/clay density, bulk and shear modulus
Dc, Kc, Gc = 2.65, 36.6, 45     # cement density, bulk and shear modulus
Db, Kb = 1, 2.2                 # brine density, bulk modulus
phi_c = 0.4                      # critical porosity
sigma = 20                       # effective pressure
scheme = 2
Cn = 8.6
vsh = 0                          # shale volume
phib = 0.3                       # cement porosity for Vs
f = 0.5                          # slip factor

# read field data (Ovelse_3_test.csv)
csv_paths = [
    Path("Excerise_1") / "Code" / "fm10_rockphysics_outputs.csv",
    Path("./Excerise_1/Code/fm10_rockphysics_outputs.csv"),
    Path("/workspaces/well_data_preprocessing/Excerise_1/Code/fm10_rockphysics_outputs.csv"),
]
csv_path = next((p for p in csv_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find 'fm10_rockphysics_outputs.csv'.")

data = pd.read_csv(csv_path)

# resolve column names (case-insensitive, common variants)
cols_lower = {c.lower(): c for c in data.columns}

def find_col(candidates, label, required=True):
    for c in candidates:
        key = c.lower()
        if key in cols_lower:
            return cols_lower[key]
    if required:
        raise KeyError(
            f"Missing {label} column. Tried {candidates}. Available: {list(data.columns)}"
        )
    return None

# Try direct Vs/phi/Vsh columns; otherwise derive from LAS curves
Vs_col = find_col(["Vs", "Vs", "PVEL", "VELP"], "Vs", required=False)
phi_col = find_col(["PHIT_ND", "PHIT", "PHI", "POR", "PHIE"], "PHI", required=False)
vsh_col = find_col(["VSH_GR", "VSH", "VSHGR"], "VSH", required=False)

if Vs_col is None or phi_col is None:
    ac_col = find_col(["AC"], "AC")
    acs_col = find_col(["ACS"], "ACS", required=False)
    den_col = find_col(["DEN", "RHOB"], "DEN")
    neu_col = find_col(["NEU", "NPHI"], "NEU")

    data = data.copy()
    data["Vs"] = (1e6 / data[ac_col]) * 0.3048
    if acs_col is not None:
        data["Vs"] = (1e6 / data[acs_col]) * 0.3048
    rho_ma = 2.65
    rho_f = 1.00
    data["phi_D"] = (rho_ma - data[den_col]) / (rho_ma - rho_f)
    data["phi_N"] = data[neu_col]
    data["phi"] = data[["phi_D", "phi_N"]].mean(axis=1)
    data.loc[(data["phi"] < 0) | (data["phi"] > 0.60), "phi"] = np.nan
    Vs_col = "Vs"
    phi_col = "phi"
    if vsh_col is None:
        vsh_col = None

# estimate cement
vcem_seeds = np.array([0, 0.005, 0.01, 0.02, 0.03, 0.04, 0.1])
phib_p = [0.3, 0.37, 0.38, 0.39, 0.395]  # cement porosity for Vs

# compute elastic bounds
phi, Vs1, Vs2, Vs3, vs1, vs2, vs3 = QI.screening(
    Dqz, Kqz, Gqz, Dsh, Ksh, Gsh, Dc, Kc, Gc, Db, Kb,
    phib, phi_c, sigma, vsh, scheme, f, Cn
 )

# create an object with data (ensure writable arrays)
Vs_arr = np.array(data[Vs_col], dtype=float, copy=True)
phi_arr = np.array(data[phi_col], dtype=float, copy=True)
if vsh_col is None:
    vsh_arr = np.zeros(len(data), dtype=float)
else:
    vsh_arr = np.array(data[vsh_col], dtype=float, copy=True)
qi = QI(Vs_arr, phi=phi_arr, Vsh=vsh_arr)

def estimate_cem_safe(Vs_values, phi_values):
    Vs_models = []
    for i in vcem_seeds:
        Vs, _VS = QI.cal_v_const(
            Dqz, Kqz, Gqz, Dsh, Ksh, Gsh, Dc, Kc, Gc, Db, Kb,
            phi_c - i, phi_c, vsh, phi_values, scheme
        )
        Vs_models.append(Vs)
    Vs_models = np.vstack(Vs_models).T  # (n_samples, n_seeds)
    vcem = np.full(len(Vs_values), np.nan, dtype=float)
    for idx, val in enumerate(Vs_values):
        arr = Vs_models[idx, :]
        order = np.argsort(arr)
        arr_sorted = arr[order]
        seeds_sorted = vcem_seeds[order]
        if val <= arr_sorted[0]:
            vcem[idx] = seeds_sorted[0]
        elif val >= arr_sorted[-1]:
            vcem[idx] = seeds_sorted[-1]
        else:
            j = np.searchsorted(arr_sorted, val)
            j = min(max(j, 1), len(arr_sorted) - 1)
            small = arr_sorted[j - 1]
            big = arr_sorted[j]
            v_small = seeds_sorted[j - 1]
            v_big = seeds_sorted[j]
            vcem[idx] = (val - small) / (big - small) * (v_big - v_small) + v_small
    return vcem

# estimate the cement volume for data (safe version)
vcem = estimate_cem_safe(Vs_arr, phi_arr)

# color-coding cement volume in the porosity and velocity cross plot
fig = qi.cement_diag_plot(
    vcem, Dqz, Kqz, Gqz, Dsh, Ksh, Gsh, Dc, Kc, Gc, Db, Kb,
    phib, phib_p, phi_c, sigma, vsh, Cn, scheme, f
 )

# make data points smaller and remove outlines
for ax in fig.axes:
    for coll in ax.collections:
        try:
            coll.set_sizes([30])
            coll.set_edgecolors("none")
            coll.set_linewidth(0)
        except Exception:
            pass
    for line in ax.lines:
        line.set_linewidth(1)

plt.ylim([1900, 6100])
plt.ylabel("Vs (Km/s)")
plt.yticks(np.arange(2000, 6200, 1000), [2, 3, 4, 5, 6])
plt.xlim(-0.01, 0.51)
plt.title("Cement volume estimation (FM-10)")
plt.show()

used_mask = np.isfinite(Vs_arr) & np.isfinite(phi_arr)
print(f"Used {used_mask.sum()} of {len(Vs_arr)} rows from the CSV.")